# 삼양애니팜 매출 × 경제지표 상관관계 분석

삼양애니팜은 감사보고서 주석에 베트남 현지 자회사(Samyang Anipharm Vietnam Co.,Ltd)를
"애견사료도소매업을 주업으로 하는 법인"으로 명시하고 있고, 본사도 원재료·제품 재고를
보유한 제조업체다 — **반려동물 사료(펫푸드) 제조** 계열사로 판단해, `samyang_food_correlation.ipynb`와
비슷하게 **곡물·축산물 원자재** 지표를 우선 대상으로 삼는다.

비상장 계열사라 분기·반기보고서가 없다 — 연 1회 감사보고서만 있어 **연 단위**로 분석한다
(표본 5개 연도, 2020년은 매출 행을 못 찾아 제외됨).

> **표본 크기 주의**: n=5로 매우 작다. 상관계수는 참고용 신호조차 되기 어렵고,
> "방향성만 보는" 수준으로 해석해야 한다.

공통 계산/시각화 함수는 `eda_utils.py`(같은 폴더)에 있다.

## 0. 환경 설정

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

sys.path.append(str(Path("../../RAG").resolve()))  # dart_parser.py가 있는 폴더
import dart_parser
import eda_utils  # 계열사 노트북 공통 로직 (같은 mandu/Eda 폴더)

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.family"] = "Malgun Gothic"  # Windows 기본 한글 폰트 (다른 OS라면 폰트명 교체)

In [ ]:
DART_DIR = Path("../../RAG/data/dart_xml")
TARGET_COMPANIES = {"(주)삼양애니팜"}

# 지표 DB(DATABASE_URL)는 Steam_Sales/dashboard/backend/.env 에 있다
DASHBOARD_ENV = Path("../../dashboard/backend/.env")
load_dotenv(DASHBOARD_ENV)

## 1. DART 공시에서 연간 매출액 추출

In [ ]:
annual_df, failed = eda_utils.extract_annual_metric(DART_DIR, TARGET_COMPANIES, value_col="revenue")
annual_df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(annual_df.index, annual_df["revenue"], marker="o", color="#3E8E5C")
ax.fill_between(annual_df.index, annual_df["revenue"], annual_df["revenue"].min() * 0.95, alpha=0.12, color="#3E8E5C")
ax.set_title("Samyang Anipharm - Annual Revenue (KRW million)")
ax.set_ylabel("KRW million")
ax.grid(alpha=0.3)
fig.autofmt_xdate()
plt.show()

## 2. DB에서 경제지표 로드

펫푸드 원재료는 곡물(옥수수/대두 등)과 육류 부산물이 큰 비중을 차지해, 곡물·축산물
지표 위주로 골랐다 (`samyang_food_correlation.ipynb`의 TARGETS와 유사).

In [ ]:
from sqlalchemy import create_engine
import os

engine = create_engine(os.environ["DATABASE_URL"], pool_pre_ping=True)

TARGETS = {
    "market_yfinance": ["옥수수", "대두", "대두유", "밀", "원달러환율"],
    "fao_food_price_index": ["Cereals", "Meat", "Food Price Index"],
    "livestock_prices": ["한우", "돼지", "닭"],
}

all_series = eda_utils.load_indicator_series(engine, TARGETS)
print(f"{len(all_series)}개 지표 시계열 로드")

## 3. 연도별 지표 평균과 매출 정렬

In [ ]:
aligned = eda_utils.align_indicators_to_periods(annual_df, "revenue", all_series)
aligned

## 4. 레벨 기준 상관관계

표본이 5개뿐이라 `min_n`을 낮춰서 계산한다.

In [ ]:
level_corr = eda_utils.corr_table(aligned, all_series, "revenue", min_n=4)
level_corr

In [ ]:
eda_utils.plot_top_correlations(level_corr, "Level correlation with Anipharm revenue (n=5, 참고용)")

## 5. 전기 대비 변화율(%) 기준 상관관계

In [ ]:
pct = aligned.drop(columns=["period_from"]).pct_change().dropna(how="all")
pct_corr = eda_utils.corr_table(pct, all_series, "revenue", min_n=3)
pct_corr

## 6. 시차(Lag) 분석

연 단위이므로 lag는 0~1년만 본다.

> **표본 크기 재차 주의**: n=5 수준이라, 여기 나온 어떤 상관계수도 확정된 관계로
> 취급하면 안 된다.

In [ ]:
level_df = aligned.drop(columns=["period_from"])
lag_df = eda_utils.lag_correlation_table(level_df, all_series, "revenue", max_lag=2, min_n=3)
lag_df

In [ ]:
eda_utils.plot_lag_heatmap(lag_df, "Lag correlation (indicator at t-L year vs revenue at t)", max_lag=2, top_n=len(lag_df))

## 결론 및 한계

*(노트북을 실행한 뒤, 위 상관관계 표를 보고 이 셀에 실제 결론을 채워 넣을 것)*

- 표본이 5개 연도뿐이라 상관계수는 통계적으로 의미 있는 수준이 아니다 — 방향성 참고용.
- 매출 자체가 연 19,500~20,600백만원 사이로 변동폭이 작아(안정적 내수 사업으로 보임),
  원자재 가격 변동이 매출보다는 원가율/마진에 먼저 반영될 가능성이 있다 — 매출 대신
  매출총이익률과 비교하면 더 뚜렷한 관계가 나올 수도 있다.